# TRAIN EyePACS v2 — canonical (all 11 models, RETFound at 15 epochs)

This is the canonical training notebook for the v2 run. It trains all eleven models; any
model already checkpointed in `02_checkpoints` is skipped.

**Epoch budget.** Every model trains for 15 epochs. v1 gave RETFound 8-10 epochs; that was
abandoned because (a) it made the architecture comparison unequal and (b) `retfound_5class`
peaked on its final epoch at 10, i.e. was still improving. With best-validation-AUC
checkpointing, extra epochs cannot select a worse model, only an equal or better one.

**Provenance.** The eight CNN / ViT-Base checkpoints were produced by
`TRAIN_eyepacs_v2_A100.ipynb` (20-21 Sept 2026), whose CNN code path is identical to this
notebook's. The RETFound checkpoints are produced by this notebook. See
`99_audit/RUN_LOG.md`.

**Reproducibility note.** `cudnn.benchmark` and mixed precision make GPU training not
bitwise deterministic across runs. The frozen float64 predictions in `03_predictions` are
the record of the models; retraining reproduces them closely but not bit-for-bit.

Prerequisites: Runtime > A100; Colab secret `HF_TOKEN` with notebook access on (RETFound
weights are gated). Run cells 1-5 in order.

In [1]:
# Cell 1: auth (Google + HuggingFace), deps, Drive helpers, RETFound preflight
!pip -q install timm transformers
import io, os, json, time, zipfile, glob, datetime, random
import numpy as np, pandas as pd, torch
import torch.nn as nn

from google.colab import auth, userdata
auth.authenticate_user()
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))
print('HF authenticated')

from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
svc = gbuild('drive', 'v3')

F_SPLITS      = '13x_CS7N_UasGRSuNYMyD5To2RTYjL-EP'
F_CHECKPOINTS = '1DAAXJ2fjk2zAzx7PpGaWsk4BXudeA5Le'
F_PREDICTIONS = '1nCloJ5o9veZb2xJL_2EjdVHQXN31jy4w'
F_AUDIT       = '1zuvfcxC3umHAq_uf7Vx9omMEgUEuQhTt'

SPLIT_IDS = {'train': '1FU30c6vRYMLqX8h9z2F58iEqow1Zx8NQ',
             'val':   '1Z9tFlAGe0j4RVRgG9-6LnhTrukvFy87y',
             'test':  '1mSlAqeNtFJ0Ap6BMenrrQxHaWep-9nEj'}
IMAGE_ZIPS = {'batch1': '11WspNNHS73OadbW-sqy7BSzcrsBkX1d6',
              'batch2': '1zG7rHjm8Ty9LHG4LqMzgnOzrT_90VO9H',
              'batch3': '1L88vUNSxWjdGPBquhknZhafcnXXRd3ES',
              'batch4': '1-FrpGB74qEGzDvXlA9FhmB2OeeZkKPAN',
              'batch5': '1mEIrSGVExZ6zmxnRLxyFab5yB1kH5YBT'}

RETFOUND_REPO = 'iszt/RETFound_mae_meh'
NUM_WORKERS = 8

def dl(fid, path):
    with io.FileIO(path, 'wb') as fh:
        d = MediaIoBaseDownload(fh, svc.files().get_media(fileId=fid), chunksize=50*1024*1024)
        done = False
        while not done: _, done = d.next_chunk()

def find_id(name, folder):
    r = svc.files().list(q=f"'{folder}' in parents and name='{name}' and trashed=false",
                         fields='files(id)').execute().get('files', [])
    return r[0]['id'] if r else None

def up(path, folder, mime='application/octet-stream', overwrite=False):
    media = MediaFileUpload(path, mimetype=mime, resumable=True)
    name = os.path.basename(path)
    if overwrite:
        fid = find_id(name, folder)
        if fid:
            svc.files().update(fileId=fid, media_body=media).execute(); return
    svc.files().create(body={'name': name, 'parents': [folder]}, media_body=media, fields='id').execute()
    print('  uploaded', name)

def existing(folder):
    names, tok = set(), None
    while True:
        r = svc.files().list(q=f"'{folder}' in parents and trashed=false",
                             fields='files(name),nextPageToken', pageToken=tok, pageSize=1000).execute()
        names |= {f['name'] for f in r.get('files', [])}
        tok = r.get('nextPageToken')
        if not tok: break
    return names

STATUS = {'started': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'events': []}
def heartbeat(**kw):
    STATUS.update(kw); STATUS['updated'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    with open('TRAINING_STATUS.json', 'w') as f: json.dump(STATUS, f, indent=2)
    try: up('TRAINING_STATUS.json', F_AUDIT, 'application/json', overwrite=True)
    except Exception as e: print('  (heartbeat failed, continuing):', e)

def find_blocks(vit):
    """Locate the transformer block ModuleList without assuming an attribute path."""
    best = None
    for _, mod in vit.named_modules():
        if isinstance(mod, nn.ModuleList) and len(mod) >= 12:
            if best is None or len(mod) > len(best):
                best = mod
    if best is None:
        raise RuntimeError('could not locate transformer blocks in the encoder')
    return best

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('CPU cores:', os.cpu_count(), '| workers:', NUM_WORKERS)

from transformers import AutoModel, AutoConfig
_cfg = AutoConfig.from_pretrained(RETFOUND_REPO)
_m = AutoModel.from_pretrained(RETFOUND_REPO, add_pooling_layer=False)
_n = sum(p.numel() for p in _m.parameters())
_blocks = find_blocks(_m)
print(f'\nRETFound preflight: hidden={_cfg.hidden_size} layers={_cfg.num_hidden_layers} params={_n/1e6:.1f}M blocks={len(_blocks)}')
assert _cfg.hidden_size == 1024 and _cfg.num_hidden_layers == 24, 'not ViT-Large'
assert _n > 2.9e8, f'parameter count {_n/1e6:.0f}M too low for ViT-L/16'
assert len(_blocks) == 24, f'found {len(_blocks)} blocks, expected 24'
print('  PREFLIGHT PASSED\n')
del _m, _blocks

HF authenticated
GPU: NVIDIA A100-SXM4-40GB
CPU cores: 12 | workers: 8


config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]


RETFound preflight: hidden=1024 layers=24 params=303.3M blocks=24
  PREFLIGHT PASSED



In [2]:
# Cell 2: roster. All models 15 epochs; other hyperparameters are v1's.
MODELS = [
  {'name':'densenet121_binary','arch':'densenet121','task':'binary','nc':1,'epochs':15,'bs':32,'lr':1e-4,'wd':1e-4},
  {'name':'densenet121_5class','arch':'densenet121','task':'5class','nc':5,'epochs':15,'bs':32,'lr':1e-4,'wd':1e-4},
  {'name':'efficientnet_b3_binary','arch':'efficientnet_b3','task':'binary','nc':1,'epochs':15,'bs':32,'lr':1e-4,'wd':1e-4},
  {'name':'efficientnet_b3_5class','arch':'efficientnet_b3','task':'5class','nc':5,'epochs':15,'bs':32,'lr':1e-4,'wd':1e-4},
  {'name':'resnet50_binary','arch':'resnet50','task':'binary','nc':1,'epochs':15,'bs':32,'lr':1e-4,'wd':1e-4},
  {'name':'resnet50_5class','arch':'resnet50','task':'5class','nc':5,'epochs':15,'bs':32,'lr':1e-4,'wd':1e-4},
  {'name':'vit_base_binary','arch':'vit_base_patch16_224','task':'binary','nc':1,'epochs':15,'bs':32,'lr':1e-4,'wd':1e-4},
  {'name':'vit_base_5class','arch':'vit_base_patch16_224','task':'5class','nc':5,'epochs':15,'bs':32,'lr':1e-4,'wd':1e-4},
  {'name':'retfound_binary','arch':'retfound','task':'binary','nc':1,'epochs':15,'bs':8,'lr':5e-5,'wd':0.05,'clip':1.0,'lldecay':0.65},
  {'name':'retfound_5class','arch':'retfound','task':'5class','nc':5,'epochs':15,'bs':8,'lr':5e-5,'wd':0.05,'clip':1.0,'lldecay':0.65},
  {'name':'retfound_adversarial','arch':'retfound','task':'adversarial_binary','nc':1,'epochs':15,'bs':8,'lr':3e-5,'wd':0.05,'clip':1.0,'lldecay':0.65,'alpha':0.08,'primary':'densenet121_binary'},
]
ALL = [m['name'] for m in MODELS]
RUN_MODELS = ALL          # finished models are skipped automatically
IMG_SIZE = 224
BASE_SEED = 20260920
print('will attempt:', RUN_MODELS)

will attempt: ['densenet121_binary', 'densenet121_5class', 'efficientnet_b3_binary', 'efficientnet_b3_5class', 'resnet50_binary', 'resnet50_5class', 'vit_base_binary', 'vit_base_5class', 'retfound_binary', 'retfound_5class', 'retfound_adversarial']


In [3]:
# Cell 3: fetch splits and images, verify coverage
os.makedirs('/content/img', exist_ok=True)
splits = {}
for s, fid in SPLIT_IDS.items():
    p = f'split_{s}.csv'
    if not os.path.exists(p): dl(fid, p)
    splits[s] = pd.read_csv(p); print(s, len(splits[s]))

heartbeat(stage='downloading images')
if not glob.glob('/content/img/**/*.jp*g', recursive=True):
    for name, fid in IMAGE_ZIPS.items():
        z = f'/content/{name}.zip'
        if not os.path.exists(z):
            print('downloading', name, '...'); dl(fid, z)
        with zipfile.ZipFile(z) as zf: zf.extractall('/content/img')
        os.remove(z); print('extracted', name)
        heartbeat(stage=f'extracted {name}')

paths = glob.glob('/content/img/**/*.jp*g', recursive=True) + glob.glob('/content/img/**/*.png', recursive=True)
INDEX = {os.path.splitext(os.path.basename(p))[0]: p for p in paths}
need = set().union(*[set(d['image_id'].astype(str)) for d in splits.values()])
missing = sorted(need - set(INDEX))
print('images on disk:', len(INDEX), '| required:', len(need), '| missing:', len(missing))
heartbeat(stage='images ready', images_on_disk=len(INDEX), images_missing=len(missing))
if missing:
    print('first missing:', missing[:10]); raise SystemExit('Images missing. Stop and report.')
print('all split images present')

train 13532
val 2900
test 2900
downloading batch1 ...
extracted batch1
downloading batch2 ...
extracted batch2
downloading batch3 ...
extracted batch3
downloading batch4 ...
extracted batch4
downloading batch5 ...
extracted batch5
images on disk: 19332 | required: 19332 | missing: 0
all split images present


In [4]:
# Cell 4: dataset, model builders, layer-wise LR decay, train, predict
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import roc_auc_score
from PIL import Image

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]
TF_TRAIN = transforms.Compose([transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.ColorJitter(0.2,0.2,0.1), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
TF_EVAL = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class DR(Dataset):
    def __init__(self, df, tf): self.df = df.reset_index(drop=True); self.tf = tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return self.tf(Image.open(INDEX[str(r['image_id'])]).convert('RGB')), int(r['level']), int(r['binary_label']), i

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

class RETFound(nn.Module):
    """RETFound MAE ViT-L/16 encoder + linear head; global average pooling over patch tokens."""
    def __init__(self, num_classes):
        super().__init__()
        self.base = AutoModel.from_pretrained(RETFOUND_REPO, add_pooling_layer=False)
        h = self.base.config.hidden_size
        self.fc_norm = nn.LayerNorm(h, eps=1e-6)
        self.classifier = nn.Linear(h, num_classes)
        nn.init.trunc_normal_(self.classifier.weight, std=0.01)
        nn.init.zeros_(self.classifier.bias)
    def forward(self, x):
        h = self.base(pixel_values=x).last_hidden_state
        return self.classifier(self.fc_norm(h[:, 1:].mean(dim=1)))

def build(cfg):
    if cfg['arch'] == 'retfound':
        m = RETFound(cfg['nc'])
        n = sum(p.numel() for p in m.base.parameters())
        if n < 2.9e8:
            raise RuntimeError(f'RETFound encoder only {n/1e6:.0f}M params — weights did not load')
        print(f'  RETFound encoder loaded ({n/1e6:.1f}M params, {len(find_blocks(m.base))} blocks)')
        STATUS['retfound_weights'] = f'loaded {n/1e6:.1f}M'
        return m
    return timm.create_model(cfg['arch'], pretrained=True, num_classes=cfg['nc'])

def make_optimizer(model, cfg):
    """Layer-wise LR decay for RETFound (0.65, head at 10x); plain AdamW otherwise."""
    if cfg['arch'] != 'retfound':
        return torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    d, lr = cfg['lldecay'], cfg['lr']
    blocks = find_blocks(model.base)
    n = len(blocks)
    block_ids = {id(p) for blk in blocks for p in blk.parameters()}
    embed = [p for nm, p in model.base.named_parameters()
             if id(p) not in block_ids and 'embed' in nm.lower()]
    embed_ids = {id(p) for p in embed}
    rest = [p for p in model.base.parameters()
            if id(p) not in block_ids and id(p) not in embed_ids]
    groups = [{'params': embed, 'lr': lr * (d ** n)}]
    for i, blk in enumerate(blocks):
        groups.append({'params': list(blk.parameters()), 'lr': lr * (d ** (n - 1 - i))})
    groups.append({'params': rest + list(model.fc_norm.parameters()), 'lr': lr})
    groups.append({'params': list(model.classifier.parameters()), 'lr': lr * 10})
    print(f'  LLRD {d}: {len(groups)} groups, {lr*(d**n):.2e} (embed) -> {lr*10:.2e} (head)')
    return torch.optim.AdamW(groups, lr=lr, weight_decay=cfg['wd'])

@torch.no_grad()
def predict(model, df, dev, bs=64):
    model.eval()
    dl_ = DataLoader(DR(df, TF_EVAL), batch_size=bs, shuffle=False,
                     num_workers=NUM_WORKERS, pin_memory=True)
    probs = np.zeros(len(df), dtype=np.float64)
    for x, _, _, idx in dl_:
        out = model(x.to(dev, non_blocking=True)).float()
        p = torch.sigmoid(out[:, 0]) if out.shape[1] == 1 else torch.softmax(out, 1)[:, 2:].sum(1)
        probs[idx.numpy()] = p.double().cpu().numpy()
    return probs

def train_one(cfg, dev, model_i, n_models, seed, eval_bs):
    seed_everything(seed)
    tr, va = splits['train'], splits['val']
    model = build(cfg).to(dev)
    primary_err = None
    if cfg['task'] == 'adversarial_binary':
        pf = f"eyepacs_predictions_train_{cfg['primary']}_v2.csv"
        if not os.path.exists(pf):
            fid = find_id(pf, F_PREDICTIONS)
            if not fid: raise RuntimeError(f"need {cfg['primary']} train predictions first")
            dl(fid, pf)
        pp = pd.read_csv(pf).set_index('image_id')['predicted_probability']
        primary_err = torch.tensor((pp.reindex(tr['image_id'].astype(str)).values >= 0.5).astype(float)
                                   - tr['binary_label'].values, dtype=torch.float32)
    g = torch.Generator(); g.manual_seed(seed)
    dl_ = DataLoader(DR(tr, TF_TRAIN), batch_size=cfg['bs'], shuffle=True,
                     num_workers=NUM_WORKERS, drop_last=True, generator=g,
                     pin_memory=True, persistent_workers=True, prefetch_factor=4)
    opt = make_optimizer(model, cfg)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg['epochs'])
    lossf = nn.BCEWithLogitsLoss() if cfg['nc'] == 1 else nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda')

    best_auc, best_state, best_ep = -1.0, None, -1
    for ep in range(cfg['epochs']):
        model.train(); tot = 0.0; t0 = time.time()
        for x, lvl, lab, idx in dl_:
            x = x.to(dev, non_blocking=True)
            y = (lab.float() if cfg['nc'] == 1 else lvl.long()).to(dev, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                out = model(x)
                loss = lossf(out[:, 0], y) if cfg['nc'] == 1 else lossf(out, y)
                if primary_err is not None:
                    e_self = torch.sigmoid(out[:, 0].float()) - lab.float().to(dev)
                    e_prim = primary_err[idx].to(dev)
                    if e_self.std() > 1e-6 and e_prim.std() > 1e-6:
                        corr = ((e_self - e_self.mean()) * (e_prim - e_prim.mean())).mean() / (e_self.std() * e_prim.std())
                        loss = loss + cfg['alpha'] * corr
            scaler.scale(loss).backward()
            if cfg.get('clip'):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['clip'])
            scaler.step(opt); scaler.update()
            tot += float(loss.item())
        sch.step()

        vp = predict(model, va, dev, eval_bs)
        vauc = float(roc_auc_score(va['binary_label'].values, vp))
        vacc = float(((vp >= 0.5).astype(int) == va['binary_label'].values).mean())
        secs = time.time() - t0; star = ''
        if vauc > best_auc:
            best_auc, best_ep = vauc, ep + 1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            star = '  <- best'
        print(f"  epoch {ep+1}/{cfg['epochs']}  loss {tot/max(1,len(dl_)):.4f}  val_auc {vauc:.4f}  val_acc {vacc:.4f}  ({secs:.0f}s){star}")
        heartbeat(stage='training', current_model=cfg['name'], model_index=f'{model_i}/{n_models}',
                  epoch=f"{ep+1}/{cfg['epochs']}", last_loss=round(tot/max(1,len(dl_)), 4),
                  last_val_auc=round(vauc, 4), last_val_acc=round(vacc, 4),
                  best_val_auc=round(best_auc, 4), best_epoch=best_ep, seconds_per_epoch=round(secs))

    model.load_state_dict(best_state)
    print(f'  restored best epoch {best_ep} (val_auc {best_auc:.4f})')
    return model, best_auc, best_ep

In [5]:
# Cell 5: run, save best checkpoint + float64 predictions for all splits
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
have = existing(F_CHECKPOINTS)
todo = [m for m in MODELS if m['name'] in RUN_MODELS and f"eyepacs_{m['name']}_v2.pt" not in have]
heartbeat(stage='starting', queued=[m['name'] for m in todo],
          already_done=sorted(n for n in ALL if f'eyepacs_{n}_v2.pt' in have))
print('to train this session:', [m['name'] for m in todo])

for i, cfg in enumerate(todo, 1):
    seed = BASE_SEED + ALL.index(cfg['name'])
    eval_bs = 32 if cfg['arch'] == 'retfound' else 64
    print(f"\n=== [{i}/{len(todo)}] {cfg['name']} (seed {seed}) ===")
    t0 = time.time()
    try:
        model, best_auc, best_ep = train_one(cfg, dev, i, len(todo), seed, eval_bs)
    except Exception as e:
        print('  FAILED:', e)
        STATUS['events'].append({'model': cfg['name'], 'status': 'failed', 'error': str(e)})
        heartbeat(stage='model failed', current_model=cfg['name']); continue

    ck = f"eyepacs_{cfg['name']}_v2.pt"
    torch.save({'state_dict': model.state_dict(), 'config': cfg, 'seed': seed,
                'best_val_auc': best_auc, 'best_epoch': best_ep}, ck)
    up(ck, F_CHECKPOINTS)

    for s in ['train', 'val', 'test']:
        d = splits[s]; pr = predict(model, d, dev, eval_bs)
        pd.DataFrame({'image_id': d['image_id'].astype(str), 'patient_id': d['patient_id'].astype(str),
                      'eye': d['eye'], 'dataset': 'eyepacs', 'split': s,
                      'true_grade': d['level'].astype(int), 'true_label': d['binary_label'].astype(int),
                      'predicted_probability': pr, 'predicted_class': (pr >= 0.5).astype(int),
                      'model_name': cfg['name']}).to_csv(
            f"eyepacs_predictions_{s}_{cfg['name']}_v2.csv", index=False, float_format='%.17g')
        up(f"eyepacs_predictions_{s}_{cfg['name']}_v2.csv", F_PREDICTIONS, 'text/csv')

    mins = round((time.time() - t0) / 60, 1)
    STATUS['events'].append({'model': cfg['name'], 'status': 'ok', 'minutes': mins,
                             'best_val_auc': round(best_auc, 4), 'best_epoch': best_ep, 'seed': seed})
    heartbeat(stage='model complete', current_model=cfg['name'])
    print(f'  done in {mins} min (best val_auc {best_auc:.4f} @ epoch {best_ep})')
    del model; torch.cuda.empty_cache()

done = existing(F_CHECKPOINTS)
remaining = [m for m in ALL if f'eyepacs_{m}_v2.pt' not in done]
heartbeat(stage='session finished', trained=len(ALL) - len(remaining), of=len(ALL), remaining=remaining)
print('\ntrained:', len(ALL) - len(remaining), '/', len(ALL), '| remaining:', remaining)

to train this session: ['retfound_binary', 'retfound_5class', 'retfound_adversarial']

=== [1/3] retfound_binary (seed 20260928) ===


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

  RETFound encoder loaded (303.3M params, 24 blocks)
  LLRD 0.65: 27 groups, 1.62e-09 (embed) -> 5.00e-04 (head)
  epoch 1/15  loss 0.4242  val_auc 0.8292  val_acc 0.8659  (182s)  <- best
  epoch 2/15  loss 0.3718  val_auc 0.8599  val_acc 0.8786  (176s)  <- best
  epoch 3/15  loss 0.3480  val_auc 0.8679  val_acc 0.8845  (176s)  <- best
  epoch 4/15  loss 0.3356  val_auc 0.8740  val_acc 0.8848  (176s)  <- best
  epoch 5/15  loss 0.3383  val_auc 0.8728  val_acc 0.8831  (176s)
  epoch 6/15  loss 0.3227  val_auc 0.8714  val_acc 0.8814  (177s)
  epoch 7/15  loss 0.3212  val_auc 0.8798  val_acc 0.8886  (175s)  <- best
  epoch 8/15  loss 0.3116  val_auc 0.8780  val_acc 0.8862  (176s)
  epoch 9/15  loss 0.3078  val_auc 0.8755  val_acc 0.8879  (177s)
  epoch 10/15  loss 0.3019  val_auc 0.8693  val_acc 0.8886  (177s)
  epoch 11/15  loss 0.2988  val_auc 0.8728  val_acc 0.8907  (177s)
  epoch 12/15  loss 0.2887  val_auc 0.8748  val_acc 0.8893  (176s)
  epoch 13/15  loss 0.2870  val_auc 0.8710  val

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

  RETFound encoder loaded (303.3M params, 24 blocks)
  LLRD 0.65: 27 groups, 1.62e-09 (embed) -> 5.00e-04 (head)
  epoch 1/15  loss 0.7706  val_auc 0.8515  val_acc 0.8597  (178s)  <- best


...

  epoch 3/15  loss 0.6432  val_auc 0.8712  val_acc 0.8817  (177s)  <- best
  epoch 4/15  loss 0.6268  val_auc 0.8654  val_acc 0.8828  (178s)
  epoch 5/15  loss 0.6129  val_auc 0.8733  val_acc 0.8855  (177s)  <- best
  epoch 6/15  loss 0.5960  val_auc 0.8756  val_acc 0.8886  (177s)  <- best
  epoch 7/15  loss 0.5797  val_auc 0.8731  val_acc 0.8869  (177s)
  epoch 8/15  loss 0.5763  val_auc 0.8696  val_acc 0.8883  (177s)
  epoch 9/15  loss 0.5570  val_auc 0.8734  val_acc 0.8893  (177s)
  epoch 10/15  loss 0.5504  val_auc 0.8738  val_acc 0.8869  (178s)
  epoch 11/15  loss 0.5341  val_auc 0.8764  val_acc 0.8866  (176s)  <- best
  epoch 12/15  loss 0.5270  val_auc 0.8736  val_acc 0.8883  (178s)
  epoch 13/15  loss 0.5208  val_auc 0.8763  val_acc 0.8872  (178s)
  epoch 14/15  loss 0.5148  val_auc 0.8746  val_acc 0.8859  (178s)
  epoch 15/15  loss 0.5115  val_auc 0.8743  val_acc 0.8862  (179s)
  restored best epoch 11 (val_auc 0.8764)
  uploaded eyepacs_retfound_5class_v2.pt
  uploaded eyepac

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

  RETFound encoder loaded (303.3M params, 24 blocks)
  LLRD 0.65: 27 groups, 9.71e-10 (embed) -> 3.00e-04 (head)
  epoch 1/15  loss 0.4481  val_auc 0.8332  val_acc 0.8628  (179s)  <- best
  epoch 2/15  loss 0.3799  val_auc 0.8578  val_acc 0.8734  (179s)  <- best
  epoch 3/15  loss 0.3690  val_auc 0.8576  val_acc 0.8800  (179s)
  epoch 4/15  loss 0.3525  val_auc 0.8633  val_acc 0.8748  (178s)  <- best
  epoch 5/15  loss 0.3461  val_auc 0.8703  val_acc 0.8831  (179s)  <- best


  epoch 6/15  loss 0.3364  val_auc 0.8717  val_acc 0.8810  (180s)  <- best


  epoch 7/15  loss 0.3388  val_auc 0.8749  val_acc 0.8834  (178s)  <- best
  epoch 8/15  loss 0.3302  val_auc 0.8723  val_acc 0.8852  (179s)
  epoch 9/15  loss 0.3196  val_auc 0.8732  val_acc 0.8879  (178s)
  epoch 10/15  loss 0.3223  val_auc 0.8756  val_acc 0.8890  (179s)  <- best
  epoch 11/15  loss 0.3105  val_auc 0.8771  val_acc 0.8890  (179s)  <- best
  epoch 12/15  loss 0.3154  val_auc 0.8756  val_acc 0.8890  (179s)
  epoch 13/15  loss 0.3119  val_auc 0.8748  val_acc 0.8893  (180s)
  epoch 14/15  loss 0.3132  val_auc 0.8747  val_acc 0.8890  (180s)
  epoch 15/15  loss 0.3126  val_auc 0.8759  val_acc 0.8907  (179s)
  restored best epoch 11 (val_auc 0.8771)
  uploaded eyepacs_retfound_adversarial_v2.pt
  uploaded eyepacs_predictions_train_retfound_adversarial_v2.csv
  uploaded eyepacs_predictions_val_retfound_adversarial_v2.csv
  uploaded eyepacs_predictions_test_retfound_adversarial_v2.csv
  done in 47.3 min (best val_auc 0.8771 @ epoch 11)

trained: 11 / 11 | remaining: []
